## MLS WS 2025/26 Exercise 3b: Network Intrusion Detection (37P)
*Adapted from an exercise created by Dennis Eisermann*

In this exercise, you classify the [CIC-IDS-2017](https://www.unb.ca/cic/datasets/ids-2017.html) cybersecurity dataset. This dataset contains network traffic data to train and evaluate [network intrusion detection systems](https://en.wikipedia.org/wiki/Intrusion_detection_system). The dataset consists of modern network traffic and contains more than 2.8 million network packets recorded over a seven-day period in a real network environment. The dataset covers both regular traffic and multiple attack scenarios like [Brute Force](https://en.wikipedia.org/wiki/Brute-force_search), [DoS](https://en.wikipedia.org/wiki/Denial-of-service_attack), and [Web Attacks](https://owasp.org/www-project-top-ten/). The data will be used for a multiclass classification task based on the feature "Attack Type". [Deep Neural Network (DNN)](https://en.wikipedia.org/wiki/Deep_learning) architectures are used to divide network flow samples into benign and different attack types.

### Setup

In [1]:
%pip install numpy pandas scikit-learn imbalanced-learn
%pip install torch --index-url https://download.pytorch.org/whl/cu126

Defaulting to user installation because normal site-packages is not writeable
     |████████████████████████████████| 258 kB 1.7 MB/s eta 0:00:01
You should consider upgrading via the '/Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.
Defaulting to user installation because normal site-packages is not writeable
Looking in indexes: https://download.pytorch.org/whl/cu126
ERROR: Could not find a version that satisfies the requirement torch (from versions: none)
ERROR: No matching distribution found for torch
You should consider upgrading via the '/Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.


Download [MachineLearningCSV.zip](http://cicresearch.ca/CICDataset/CIC-IDS-2017/Dataset/CIC-IDS-2017/CSVs/MachineLearningCSV.zip) version from the CIC-IDS-2017 dataset.

### Task 1 – Exploration of the Dataset (14 Points)

Create an understanding of the dataset. Additionally, we want a more in-depth understanding of the attack classes and their features to get an idea of how a classifier could work.

1. Unzip MachineLearningCSV.zip and load all contained CSV files and concatenate them into one dataframe. Show the top 5 rows for a first impression of the dataset. (1 Point)

In [4]:
from pathlib import Path
import zipfile
import pandas as pd

#concatenate csv files from ../data/MachineLearningCVE into one dataframe
data_dir = Path("../data/MachineLearningCVE")
all_files = data_dir.glob("*.csv")
df_list = [pd.read_csv(file) for file in all_files]
df = pd.concat(df_list, ignore_index=True)

                 


2. Print the column statistics and data types. (1 Point)

In [16]:
# print column statistics and data types
df.describe(include='all')


/Users/uwagstev/Library/Python/3.9/lib/python/site-packages/pandas/core/nanops.py:1016: RuntimeWarning: invalid value encountered in subtract
  sqr = _ensure_numeric((avg - values) ** 2)
/Users/uwagstev/Library/Python/3.9/lib/python/site-packages/pandas/core/nanops.py:1016: RuntimeWarning: invalid value encountered in subtract
  sqr = _ensure_numeric((avg - values) ** 2)


,Destination Port,Flow Duration,Total Fwd Packets,Total Backward Packets,Total Length of Fwd Packets,Total Length of Bwd Packets,Fwd Packet Length Max,Fwd Packet Length Min,Fwd Packet Length Mean,Fwd Packet Length Std,...,min_seg_size_forward,Active Mean,Active Std,Active Max,Active Min,Idle Mean,Idle Std,Idle Max,Idle Min,Label
count,2.830743e+06,2.830743e+06,2.830743e+06,2.830743e+06,2.830743e+06,2.830743e+06,2.830743e+06,2.830743e+06,2.830743e+06,2.830743e+06,...,2.830743e+06,2.830743e+06,2.830743e+06,2.830743e+06,2.830743e+06,2.830743e+06,2.830743e+06,2.830743e+06,2.830743e+06,2830743
unique,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,15
top,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,BENIGN
freq,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2273097
mean,8.071483e+03,1.478566e+07,9.361160e+00,1.039377e+01,5.493024e+02,1.616264e+04,2.075999e+02,1.871366e+01,5.820194e+01,6.891013e+01,...,-2.741688e+03,8.155132e+04,4.113412e+04,1.531825e+05,5.829582e+04,8.316037e+06,5.038439e+05,8.695752e+06,7.920031e+06,NaN
std,1.828363e+04,3.365374e+07,7.496728e+02,9.973883e+02,9.993589e+03,2.263088e+06,7.171848e+02,6.033935e+01,1.860912e+02,2.811871e+02,...,1.084989e+06,6.485999e+05,3.933815e+05,1.025825e+06,5.770923e+05,2.363008e+07,4.602984e+06,2.436689e+07,2.336342e+07,NaN
min,0.000000e+00,-1.300000e+01,1.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,...,-5.368707e+08,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,NaN
25%,5.300000e+01,1.550000e+02,2.000000e+00,1.000000e+00,1.200000e+01,0.000000e+00,6.000000e+00,0.000000e+00,6.000000e+00,0.000000e+00,...,2.000000e+01,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,NaN
50%,8.000000e+01,3.131600e+04,2.000000e+00,2.000000e+00,6.200000e+01,1.230000e+02,3.700000e+01,2.000000e+00,3.400000e+01,0.000000e+00,...,2.400000e+01,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,NaN
75%,4.430000e+02,3.204828e+06,5.000000e+00,4.000000e+00,1.870000e+02,4.820000e+02,8.100000e+01,3.600000e+01,5.000000e+01,2.616295e+01,...,3.200000e+01,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,NaN


In [15]:

print(df.dtypes)

 Destination Port                int64
 Flow Duration                   int64
 Total Fwd Packets               int64
 Total Backward Packets          int64
Total Length of Fwd Packets      int64
                                ...   
Idle Mean                      float64
 Idle Std                      float64
 Idle Max                        int64
 Idle Min                        int64
 Label                          object
Length: 79, dtype: object


3. Do you notice something odd in the statistics for some columns? Print all column names that have overall values that might be a problem for training a classifier. (3 Points)
   *Note: There are three different types of "oddities" you show be able to find in the statistics.*

In [17]:
# print columns that contain NaN values
nan_columns = df.columns[df.isna().any()].tolist()
print("Columns with NaN values:", nan_columns)

Columns with NaN values: ['Flow Bytes/s']


4. Show how many different values each column contains. (1 Point)

5. Count the duplicate rows in as total numbers and as percentage. (1 Points)

6. Which column contains information about the kind of attack? Plot the number of rows for each attack. (1 Point)
   *Note: There are white spaces in some of the column names.*

7. Show the distribution of attacks in a pie chart with the value labels in the plot formatted as percentage. Attacks less frequent than 4% of the data should be aggregated into a single bucket called "other". (2 Points)

In [ ]:
import matplotlib.pyplot as plt


8. Visualize the correlations between numeric features. (1 Point)

9. Do you recognize the columns that produce the black stripes? Answer where do you know them from and what they have in common. (1 Point)

*your answer*

10. Besides the columns from subtask 9, you identified two other columns in subtask 3 that contain odd values. Research, why the dataset contains this kind of values. It is helpful to know, that the creators of the dataset used [CICFlowMeter](https://github.com/ahlashkari/CICFlowMeter/tree/master) for the measurements. You might also be interested in a paper related to this [topic](https://intrusion-detection.distrinet-research.be/WTMC2021/Resources/wtmc2021_Engelen_Troubleshooting.pdf). (2 Points)

*your answer*

### Task 2 – Cleaning the Dataset (7 Points)
This dataset has to be [cleaned](https://en.wikipedia.org/wiki/Data_cleansing), before we can use it.



1. Clean the white spaces from the column labels. (1 Point)

2. Treat the columns with the "odd" values from task 1, subtask 3, each in its own sensible way. Thus, first answer for each of the three types of unusable values, how you want to treat them and why this is a prudent solution. (3 Points)

*your answer*

3. Perform the cleanup you proposed in subtask 2 for all three columns. (3 Points)

### Task 3 – Preprocessing (10 Points)

Your goal in this task is to prepare the dataset to train a Deep Neural Network. Use the knowledge you gained in the exploration phase.

1. Reduce Memory usage by downcasting numeric values. Print the usage before and after the manipulation. (2 Points)

In [110]:
start_mem = df.memory_usage(deep=True).sum() / 1024**2


Memory usage decreased from 449.16 MB to 235.12 MB (47.7% reduction)


2. Research the term "stratification" and explain in two sentences what could happen when not stratifying the training/validation/test splits of a dataset. (2 Points)

*your answer*

3. Split the the dataset for training, validation, and testing in the ratio (80/10/10) with a Random State of 0. Stratify the splits on the label. (2 Points)

In [147]:
from sklearn.model_selection import train_test_split

label = 'Label'
full_dataset = data.drop(label, axis = 1)
targets = data[label]


4. Shrink the largest class in the training set with [Random Under Sampling](https://imbalanced-learn.org/stable/references/generated/imblearn.under_sampling.RandomUnderSampler.html). Enlarge all other classes with [Random Oversampling](https://imbalanced-learn.org/dev/references/generated/imblearn.over_sampling.RandomOverSampler.html). Use random state equals zero. In practice, you can use [Synthetic Minority Over-Sampling Technique](https://imbalanced-learn.org/dev/references/generated/imblearn.over_sampling.SMOTE.html) for better results with more computational power. (1 Point)

In [148]:
from imblearn.under_sampling import RandomUnderSampler
from imblearn.over_sampling import RandomOverSampler


5. Instead of a normalization, which you did for the image classification task, here you should scale the dataset inputs for the Neural Network. (1 Point)

In [149]:
from sklearn.preprocessing import StandardScaler


6. Convert the dataset splits (likely they are numpy values) into tensors. (1 Point)

7. Build a PyTorch dataset and dataloader. Shuffle the training dataset and use a 256 as batch size. (1 Point)

In [151]:
from torch.utils.data import TensorDataset, DataLoader


### Task 3 – Model Training and Validation (6 Points)

Train a Deep Neural Network to predict the attack classes.

In [111]:
import torch.nn as nn

class SmallIDSNet(nn.Module):
    def __init__(self, input_dim: int, num_classes: int):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 64),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(32, 16),
            nn.ReLU(),
            nn.Linear(16, num_classes)
        )

    def forward(self, x):
        return self.net(x)

1. Briefly explain the architecture of the network above. (2 Points)

*your answer*

2. Create the model, the optimizer and the `BCEWithLogitsLoss` loss function. (1 Point)

In [152]:
LR          = 1e-5
EPOCHS      = 20
BATCH_SIZE  = 128

In [154]:
import torch.optim as optim

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

3. Train the model and validate the results for every epoch. (2 Points)

In [ ]:
for epoch in range(1, EPOCHS+1):

    # your code

    print(f"Epoch {epoch:02d}")
    print(f"Train Loss: {avg_loss:.4f}")
    print(f"Train Acc:{training_accuracy:.4f}")
    print(f"Validation Acc: {val_accuracy:.4f}")
    print()

4. Save the trained model and the three splits of the dataset. (1 Point)